# Lecture 6 — Class Exercise
## Choropleth Maps

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [2]:
# Task 1
# YOUR CODE HERE
# Load the built-in Gapminder dataset and select 2007
gm = px.data.gapminder()
gm_2007 = gm.loc[gm["year"] == 2007].copy()

# Calculate the global mean and each country's deviation from it
global_mean_life_exp = gm_2007["lifeExp"].mean()
gm_2007["lifeExp_vs_global_avg"] = (
    gm_2007["lifeExp"] - global_mean_life_exp
)

# Find the continent with the lowest mean life expectancy
continent_means = (
    gm_2007.groupby("continent", as_index=False)["lifeExp"]
    .mean()
    .sort_values("lifeExp")
)
lowest_continent = continent_means.iloc[0]["continent"]
lowest_continent_mean = continent_means.iloc[0]["lifeExp"]

print(f"Global average life expectancy in 2007: {global_mean_life_exp:.2f} years")
print(f"Lowest continental average: {lowest_continent} ({lowest_continent_mean:.2f} years)")


Global average life expectancy in 2007: 67.01 years
Lowest continental average: Africa (54.81 years)


In [3]:
# Diverging choropleth: zero represents the global average
fig = px.choropleth(
    gm_2007,
    locations="iso_alpha",
    locationmode="ISO-3",
    color="lifeExp_vs_global_avg",
    hover_name="country",
    hover_data={
        "lifeExp": ":.2f",
        "lifeExp_vs_global_avg": ":+.2f",
        "iso_alpha": False
    },
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    labels={
        "lifeExp": "Life expectancy (years)",
        "lifeExp_vs_global_avg": "Difference from global average (years)"
    },
    title=(
        f"{lowest_continent} has the lowest average life expectancy in 2007 "
        f"— {lowest_continent_mean:.1f} years"
    ),
    height=550,
    width=1000
)

fig.update_layout(
    font=dict(family="Arial", size=12),
    geo=dict(showframe=False, showcoastlines=True),
    coloraxis_colorbar=dict(
        title="Years vs global average",
        thickness=18
    ),
    margin=dict(l=0, r=0, t=75, b=0)
)

fig.show()


## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** *(URL or description)*

**Chart type chosen** (`px.choropleth` or `px.choropleth_mapbox`) **and reason:**

*Your answer here*

**Colour scale chosen** (sequential or diverging) **and reason:**

*Your answer here*


In [7]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
import json

from pyproj import Geod
from shapely.geometry import shape

In [8]:
# Gujarat district GeoJSON URL

geojson_url = (
    "https://services7.arcgis.com/KyVYEOdteN9vDcGi/"
    "ArcGIS/rest/services/Gujarat_WFL1/FeatureServer/1/query"
    "?where=1%3D1&outFields=NAME%2CSTATE"
    "&returnGeometry=true&outSR=4326&f=geojson"
)

# Download GeoJSON
response = requests.get(geojson_url, timeout=60)
response.raise_for_status()

# Convert response into JSON
gujarat_geojson = response.json()

print("GeoJSON downloaded successfully!")
print("Number of features:", len(gujarat_geojson["features"]))

GeoJSON downloaded successfully!
Number of features: 33


In [9]:
# Inspect the first GeoJSON feature

print(gujarat_geojson["features"][0]["properties"])

{'NAME': 'Ahmadabad', 'STATE': 'Gujarat'}


In [10]:
featureidkey="properties.NAME"

In [16]:
# Create a temporary dataframe of district names

district_names = [
    feature["properties"]["NAME"]
    for feature in gujarat_geojson["features"]
    if feature["properties"].get("NAME")
]

boundary_df = pd.DataFrame({
    "District": district_names,
    "Value": [1] * len(district_names)
})

# Plot district boundaries
fig = px.choropleth(
    data_frame=boundary_df,
    geojson=gujarat_geojson,
    locations="District",
    featureidkey="properties.NAME",
    color="Value",
    color_continuous_scale="Blues",
    fitbounds="locations",
    hover_name="District",
    title="Geographical boundaries of Gujarat districts"
)

# Hide colour scale
fig.update_layout(
    geo=dict(showframe=False, showcoastlines=True),
    coloraxis_showscale=False,
    height=700,
    width=1000,
    margin=dict(l=0, r=0, t=60, b=0)
)

# Add district names inside the map
for feature in gujarat_geojson["features"]:

    district_name = feature["properties"].get("NAME")
    geometry = feature.get("geometry")

    if not district_name or not geometry:
        continue

    # Calculate approximate centre of district
    district_shape = shape(geometry)
    center = district_shape.representative_point()

    lon = center.x
    lat = center.y

    # Add district name as text
    fig.add_trace(
        go.Scattergeo(
            lon=[lon],
            lat=[lat],
            text=[district_name],
            mode="text",
            textfont=dict(
                size=9,
                color="black"
            ),
            showlegend=False,
            hoverinfo="skip"
        )
    )

# Display map
fig.show()

In [12]:
# Initialize geodesic calculator
geod = Geod(ellps="WGS84")

district_rows = []

for feature in gujarat_geojson["features"]:

    properties = feature["properties"]

    district_name = properties.get("NAME")
    state_name = properties.get("STATE")

    if not district_name:
        continue

    if state_name and state_name.strip().lower() != "gujarat":
        continue

    geometry = feature.get("geometry")

    if geometry is None:
        continue

    # Convert GeoJSON geometry into a shape
    district_shape = shape(geometry)

    # Calculate area in square metres
    area_m2, _ = geod.geometry_area_perimeter(district_shape)

    # Convert square metres into square kilometres
    area_km2 = abs(area_m2) / 1000000

    district_rows.append({
        "District": district_name,
        "Area_km2": area_km2
    })

# Create dataframe
gujarat_df = pd.DataFrame(district_rows)

# Remove duplicate districts
gujarat_df = gujarat_df.drop_duplicates(
    subset="District"
).reset_index(drop=True)

print("District areas calculated successfully!")
print("Total districts:", len(gujarat_df))

District areas calculated successfully!
Total districts: 33


In [13]:
# Sort districts by land area
gujarat_df = gujarat_df.sort_values(
    by="Area_km2",
    ascending=False
)

# Display district information
display(gujarat_df)

,District,Area_km2
32,Kachchh,43465.867230
24,Banas Kantha,10752.541449
3,Surendranagar,8976.467468
5,Bhavnagar,8138.445311
16,Rajkot,7795.785841
21,Amreli,7385.339217
0,Ahmadabad,7337.461616
4,Bharuch,6558.694283
14,Jamnagar,6388.102333
27,Patan,5778.380251


In [14]:
# Find largest district
largest_district = gujarat_df.loc[
    gujarat_df["Area_km2"].idxmax()
]

# Find smallest district
smallest_district = gujarat_df.loc[
    gujarat_df["Area_km2"].idxmin()
]

print(
    "Largest district:",
    largest_district["District"],
    "-",
    round(largest_district["Area_km2"], 2),
    "km²"
)

print(
    "Smallest district:",
    smallest_district["District"],
    "-",
    round(smallest_district["Area_km2"], 2),
    "km²"
)

Largest district: Kachchh - 43465.87 km²
Smallest district: Dang - 1764.43 km²


In [15]:
# Gujarat district choropleth using custom GeoJSON

fig = px.choropleth(
    data_frame=gujarat_df,
    geojson=gujarat_geojson,
    locations="District",
    featureidkey="properties.NAME",
    color="Area_km2",
    color_continuous_scale="YlOrRd",
    fitbounds="locations",

    hover_name="District",

    hover_data={
        "District": False,
        "Area_km2": ":,.1f"
    },

    labels={
        "Area_km2": "Land area (km²)"
    },

    title=(
        f"{largest_district['District']} is Gujarat's largest district "
        f"by land area ({largest_district['Area_km2']:,.0f} km²)"
    ),

    height=650,
    width=1000
)

# Customize map appearance

fig.update_layout(
    font=dict(
        family="Arial",
        size=12
    ),

    geo=dict(
        showframe=False,
        showcoastlines=True,
        showcountries=False
    ),

    coloraxis_colorbar=dict(
        title="Land area (km²)",
        thickness=18
    ),

    margin=dict(
        l=0,
        r=0,
        t=75,
        b=0
    )
)

# Display final map
fig.show()